In [1]:
"""
Experimental Evaluation of Generative and Probabilistic Models
Author: Shantanu Kulkarni
Date: February 2026

This project investigates:

1. Markov Chain based text generation (Generative modeling)
2. Prompt reasoning strategies (Interview, CoT, ToT)
3. Zero-shot vs Few-shot prompting comparison
4. Gaussian Mixture Models for probabilistic clustering
5. Analytical evaluation and interpretation
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from collections import defaultdict
import random
import json
import os
import time

# ============================================================
# SECTION 1: MARKOV TEXT GENERATION EXPERIMENT
# ============================================================

class MarkovTextExperiment:

    def __init__(self, order=2):
        self.order = order
        self.model = defaultdict(list)

    def train(self, corpus):
        words = corpus.split()
        for i in range(len(words) - self.order):
            state = tuple(words[i:i+self.order])
            next_word = words[i+self.order]
            self.model[state].append(next_word)

    def generate(self, length=40):
        state = random.choice(list(self.model.keys()))
        output = list(state)

        for _ in range(length - self.order):
            if state not in self.model:
                break
            next_word = random.choice(self.model[state])
            output.append(next_word)
            state = tuple(output[-self.order:])
        return " ".join(output)

    def evaluate_complexity(self):
        states = len(self.model)
        transitions = sum(len(v) for v in self.model.values())
        return {
            "states": states,
            "transitions": transitions,
            "avg_transitions": transitions / states
        }


def run_markov_experiment():

    print("\nRunning Markov Chain Experiment")

    text = """
    Artificial intelligence is transforming industries.
    Industries rely on data for decision making.
    Data driven systems improve efficiency.
    Efficiency leads to scalability and growth.
    """

    results = {}

    for k in [1, 2, 3]:
        model = MarkovTextExperiment(order=k)
        model.train(text)
        generated = model.generate()
        stats = model.evaluate_complexity()

        print(f"\nOrder {k} Generated Text:\n{generated}")

        results[f"order_{k}"] = stats

    return results


# ============================================================
# SECTION 2: PROMPT REASONING STRATEGY EVALUATION
# ============================================================

class ReasoningStrategyEvaluator:

    def __init__(self):
        pass

    def zero_shot(self, problem):
        return f"Solve directly: {problem}"

    def chain_of_thought(self, problem):
        return f"""
Problem: {problem}

Let's reason step by step:
1. Identify unknowns
2. Use given constraints
3. Form equations
4. Solve logically
"""

    def tree_of_thought(self, problem):
        return f"""
Problem: {problem}

Approach 1: Algebraic
Approach 2: Logical deduction
Approach 3: Verification

Compare approaches and choose best.
"""

    def interview_style(self, problem):
        return f"""
Problem: {problem}

Before solving:
- What do we know?
- What is required?
- Any assumptions?
- What equations apply?
"""

    def analyze_prompt(self, prompt):
        return {
            "length": len(prompt),
            "structure_depth": prompt.count("\n"),
            "reasoning_cues": prompt.lower().count("step")
        }


def run_prompt_experiment():

    print("\nRunning Prompt Engineering Experiment")

    problem = "A farmer has 20 animals and 56 legs. How many chickens and cows?"

    evaluator = ReasoningStrategyEvaluator()

    prompts = {
        "zero_shot": evaluator.zero_shot(problem),
        "cot": evaluator.chain_of_thought(problem),
        "tot": evaluator.tree_of_thought(problem),
        "interview": evaluator.interview_style(problem)
    }

    metrics = {}

    for key, prompt in prompts.items():
        analysis = evaluator.analyze_prompt(prompt)
        print(f"\n{key.upper()} Prompt:\n{prompt}")
        metrics[key] = analysis

    return metrics


# ============================================================
# SECTION 3: GAUSSIAN MIXTURE MODEL EXPERIMENT
# ============================================================

class ProbabilisticClusteringExperiment:

    def __init__(self, components=3):
        self.components = components

    def generate_data(self):
        X, _ = make_blobs(n_samples=400, centers=self.components,
                          cluster_std=1.2, random_state=42)
        return X

    def train(self, X):
        model = GaussianMixture(n_components=self.components,
                                covariance_type='full',
                                random_state=42)
        model.fit(X)
        labels = model.predict(X)
        score = silhouette_score(X, labels)
        return model, labels, score

    def visualize(self, X, labels):
        plt.figure(figsize=(6, 5))
        plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis')
        plt.title("Gaussian Mixture Clustering")
        plt.grid(True)
        plt.savefig("gmm_result.png")
        plt.close()


def run_gmm_experiment():

    print("\nRunning Gaussian Mixture Model Experiment")

    experiment = ProbabilisticClusteringExperiment(components=3)
    X = experiment.generate_data()
    model, labels, score = experiment.train(X)

    print(f"Silhouette Score: {score:.4f}")

    experiment.visualize(X, labels)

    interpretation = "Clusters are well separated." if score > 0.5 else "Clusters show moderate overlap."

    return {
        "silhouette_score": score,
        "interpretation": interpretation
    }


# ============================================================
# SECTION 4: SAVE RESULTS
# ============================================================

def save_results(markov_results, prompt_results, gmm_results):

    all_results = {
        "markov": markov_results,
        "prompt_engineering": prompt_results,
        "gmm": gmm_results
    }

    with open("experiment_results.json", "w") as f:
        json.dump(all_results, f, indent=4)

    print("\nResults saved to experiment_results.json")


# ============================================================
# MAIN EXECUTION
# ============================================================

def main():

    start = time.time()

    markov_results = run_markov_experiment()
    prompt_results = run_prompt_experiment()
    gmm_results = run_gmm_experiment()

    save_results(markov_results, prompt_results, gmm_results)

    end = time.time()
    print(f"\nTotal runtime: {end-start:.2f} seconds")


if __name__ == "__main__":
    main()


Running Markov Chain Experiment

Order 1 Generated Text:
industries. Industries rely on data for decision making. Data driven systems improve efficiency. Efficiency leads to scalability and growth.

Order 2 Generated Text:
systems improve efficiency. Efficiency leads to scalability and growth.

Order 3 Generated Text:
Artificial intelligence is transforming industries. Industries rely on data for decision making. Data driven systems improve efficiency. Efficiency leads to scalability and growth.

Running Prompt Engineering Experiment

ZERO_SHOT Prompt:
Solve directly: A farmer has 20 animals and 56 legs. How many chickens and cows?

COT Prompt:

Problem: A farmer has 20 animals and 56 legs. How many chickens and cows?

Let's reason step by step:
1. Identify unknowns
2. Use given constraints
3. Form equations
4. Solve logically


TOT Prompt:

Problem: A farmer has 20 animals and 56 legs. How many chickens and cows?

Approach 1: Algebraic
Approach 2: Logical deduction
Approach 3: Verifi